In [ ]:
import os
import numpy as np

from skimage.io import imread
from skimage import img_as_float
from skimage.segmentation import slic
from skimage.color import rgb2hsv

from scipy.stats import circmean, circvar
from sklearn.cluster import KMeans



def load_image(path):
    img = img_as_float(imread(path))

    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)

    if img.shape[-1] == 4:
        img = img[:, :, :3]

    return img



def get_segment_features(image, segments):

    hsv = rgb2hsv(image)

    rgb_means = []
    hsv_means = []

    for seg_id in np.unique(segments):

        mask = segments == seg_id

        rgb_pixels = image[mask]
        hsv_pixels = hsv[mask]

        rgb_means.append(np.mean(rgb_pixels, axis=0))

        h = circmean(hsv_pixels[:, 0], high=1, low=0)
        s = np.mean(hsv_pixels[:, 1])
        v = np.mean(hsv_pixels[:, 2])

        hsv_means.append([h, s, v])

    return np.array(rgb_means), np.array(hsv_means)


def calculate_variances(means):
    if len(means) < 2:
        return (0.0, 0.0, 0.0)
    return np.var(means, axis=0)


def calculate_hsv_variances(means):
    if len(means) < 2:
        return (0.0, 0.0, 0.0)

    means = np.array(means)

    h_var = circvar(means[:, 0], high=1, low=0)
    s_var = np.var(means[:, 1])
    v_var = np.var(means[:, 2])

    return (h_var, s_var, v_var)


def color_dominance(image, clusters=5):

    hsv = rgb2hsv(image)
    flat = hsv.reshape(-1, 3)

    kmeans = KMeans(n_clusters=clusters, n_init=10, random_state=0)
    kmeans.fit(flat)

    colors = kmeans.cluster_centers_

    counts = np.bincount(kmeans.labels_)
    ratios = counts / len(kmeans.labels_)

    return sorted(zip(ratios, colors), key=lambda x: x[0], reverse=True)



folder = "images/"
results = []

for filename in os.listdir(folder):

    if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    path = os.path.join(folder, filename)


    image = load_image(path)

    segments = slic(image, n_segments=100, compactness=10, start_label=1)

    rgb_m, hsv_m = get_segment_features(image, segments)

    rgb_v = calculate_variances(rgb_m)
    hsv_v = calculate_hsv_variances(hsv_m)

    dom_colors = color_dominance(image)

    results.append({
        "file": filename,
        "rgb_means": rgb_m,
        "hsv_means": hsv_m,
        "rgb_var": rgb_v,
        "hsv_var": hsv_v,
        "dominant_colors": dom_colors
    })





KeyboardInterrupt: 